# Pandas transaction analysis

## Learning goals

Inspect a transaction CSV, then practise `loc`, `iloc`, missing and duplicate checks, `groupby` aggregation, `merge`, deterministic sorting, datetime parsing, transformations, and categorical dtypes on a separate synthetic settlement/provider schema.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("datasets/transactions.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError:
        print("Banking dataset not present; completed examples use settlement data.")
    else:
        uploaded = files.upload()
        if uploaded:
            DATA_PATH = Path(next(iter(uploaded)))

In [ ]:
if DATA_PATH.exists():
    transactions = pd.read_csv(DATA_PATH)
    transaction_inspection = {
        "shape": transactions.shape,
        "columns": list(transactions.columns),
        "dtypes": transactions.dtypes.astype(str).to_dict(),
        "preview": transactions.head(3),
    }
else:
    transactions = None
    transaction_inspection = {"status": "CSV not loaded; run the setup cell to choose it."}
transaction_inspection

In [ ]:
settlements = pd.DataFrame(
    {
        "batch_ref": ["set-01", "set-02", "set-02"],
        "provider_code": ["north-pay", "west-pay", "west-pay"],
        "gross_text": ["120.50", "unknown", "80.00"],
        "submitted_at": [
            "2025-02-01T08:00:00+00:00",
            "2025-02-02T09:30:00+00:00",
            "2025-02-02T09:30:00+00:00",
        ],
        "settlement_state": ["completed", None, None],
    }
)
provider_directory = pd.DataFrame(
    {
        "provider_code": ["north-pay", "west-pay"],
        "provider_region": ["north", "west"],
    }
)
settlements.head(), provider_directory

In [ ]:
settlement_view = settlements.loc[
    :, ["batch_ref", "provider_code", "gross_text", "submitted_at", "settlement_state"]
].copy()
settlement_view["gross_amount"] = pd.to_numeric(
    settlement_view["gross_text"], errors="coerce"
)
settlement_view["submitted_at"] = pd.to_datetime(
    settlement_view["submitted_at"], utc=True
)
settlement_view["settlement_state"] = settlement_view["settlement_state"].astype(
    "category"
)
first_two_batches = settlement_view.iloc[:2]
missing_states = settlement_view["settlement_state"].isna().sum()
duplicate_batches = settlement_view.duplicated(subset=["batch_ref"], keep=False)
settlement_summary = (
    settlement_view.groupby("provider_code", as_index=False)
    .agg(batch_count=("batch_ref", "size"), total_gross=("gross_amount", "sum"))
    .merge(provider_directory, on="provider_code", how="left", validate="many_to_one")
    .sort_values(["provider_region", "provider_code"], kind="stable")
)
first_two_batches, missing_states, duplicate_batches, settlement_summary

In [ ]:
# Exercise 1 — transaction quality contract.
# Report missing merchant_id values, currency codes that are not exactly three uppercase letters,
# and each composite duplicate on (account_id, occurred_at, amount_minor), without mutating input.
raise NotImplementedError("Complete this notebook exercise")

In [ ]:
# Exercise 2 — daily channel summary.
# Parse occurred_at, derive a UTC day, then group by day and channel with named aggregations
# for payment count and approved amount. Return a stable day/channel sort.
raise NotImplementedError("Complete this notebook exercise")

In [ ]:
# Exercise 3 — merchant profile enrichment.
# Left-join merchant_profiles on merchant_id with validate="many_to_one". Preserve unmatched
# transactions and original row order while attaching merchant segment and category fields.
raise NotImplementedError("Complete this notebook exercise")

## Self-check

1. When should you choose `loc` rather than `iloc`?
2. Why make duplicate handling explicit before aggregation?
3. What does `validate="many_to_one"` protect?
4. Why parse datetime values with `utc=True`?
5. When does a categorical column reduce ambiguity?